# LANDR v2 — run in your browser (no install)

Fatigue-aware ACL injury-risk screening. Runs the basic screen, the
**fatigue-vulnerability** comparison, the **accuracy benchmark**, your own jump
videos (single + two-view), and a **maximum-accuracy** mode.

Open at [colab.research.google.com](https://colab.research.google.com), then run
cells in order. Upload `LANDR_codebase_v2.zip` when asked.

## Step 1 — Install libraries

In [ ]:
!pip install -q numpy scipy matplotlib
print('✅ installed')

## Step 2 — Upload the code (choose LANDR_codebase_v2.zip)

In [ ]:
from google.colab import files
import zipfile, os
uploaded = files.upload()
zip_name = next(n for n in uploaded if n.endswith('.zip'))
with zipfile.ZipFile(zip_name) as z:
    z.extractall('.')
print('✅ unzipped:', os.listdir('landr'))

## Step 3 — Basic screen (synthetic high-risk landing)

In [ ]:
%cd landr
!python -m landr.cli demo --quality poor --plot landr_poor.png
%cd ..
from IPython.display import Image
Image('landr/landr_poor.png')

## Step 4 — Fatigue-vulnerability (Pillar 2)

In [ ]:
%cd landr
!python -m landr.cli fatigue --plot landr_fatigue.png
%cd ..
from IPython.display import Image
Image('landr/landr_fatigue.png')

## Step 5 — Accuracy benchmark (Pillar 1)

In [ ]:
%cd landr
!python -m landr.cli benchmark
%cd ..

## Step 6 — Analyze YOUR jump (single front video)
Uses the `full` pose model + refinement by default. First run downloads a model.

In [ ]:
!pip install -q mediapipe opencv-python
from google.colab import files
import os
up = files.upload()                      # FRONT-view jump clip
video = os.path.abspath(list(up.keys())[-1])
%cd landr
!python -m landr.cli analyze "$video" --accuracy balanced --plot real.png --out real.json
%cd ..
from IPython.display import Image
Image('landr/real.png')

## Step 7 — Two-view: front + side (recommended)
Film two clips of the same jump (one facing the camera, one from the side).
LANDR fuses them: **valgus from front, knee/trunk flexion from side**. Upload
front first, then side.

In [ ]:
from google.colab import files
import os
print('Upload FRONT clip:'); f = files.upload(); front = os.path.abspath(list(f.keys())[-1])
print('Upload SIDE clip:');  s = files.upload(); side  = os.path.abspath(list(s.keys())[-1])
%cd landr
!python -m landr.cli analyze2 "$front" "$side" --accuracy balanced --plot two_view.png --out two_view.json
%cd ..
from IPython.display import Image
Image('landr/two_view.png')

## Step 8 — MAXIMUM accuracy (RTMPose + two-view + heavy)
The most accurate camera-only configuration: the RTMPose backend (state-of-the-art
2D keypoints, used by Sports2D) on both views, heavy model, refinement on. Slower.
Re-run Step 7's upload first if you need to re-pick the clips, or reuse `front`/`side`.

In [ ]:
!pip install -q rtmlib onnxruntime
%cd landr
!python -m landr.cli analyze2 "$front" "$side" --backend rtmpose --accuracy max --plot max.png --out max.json
%cd ..
from IPython.display import Image
Image('landr/max.png')